# 🔄 Project 11.2 — Rotated Array Navigator
**DSA for Mechatronics · Week 11 — Searching Algorithms**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib, bisect, time
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A circular buffer in an embedded controller stores sensor calibration values.
Due to a ring-buffer wrap-around, the sorted array has been **rotated** by an
unknown offset — the smallest element is no longer at index 0.

We need to:
1. **Find the rotation point** (index of minimum) in O(log n)
2. **Search for a target** in the rotated array in O(log n) — without unrotating
3. Handle the **duplicates** variant where equal values make the pivot ambiguous
4. Apply **exponential search** for an unbounded/stream scenario


## Step 1 — Generate rotated sorted arrays

In [ ]:
# ── Personalised parameters ──────────────────────────────────────
N_ARR      = random.randint(15, 25)
VAL_RANGE  = random.randint(200, 400)
ROTATION   = random.randint(3, N_ARR - 3)   # rotate by this many positions

# Unique sorted array then rotate
base_sorted = sorted(random.sample(range(1, VAL_RANGE + 1), N_ARR))
ROTATED     = base_sorted[ROTATION:] + base_sorted[:ROTATION]

# Duplicates variant (some values repeated)
dup_base    = sorted([random.randint(1, VAL_RANGE // 3) for _ in range(N_ARR)])
ROT_DUP     = dup_base[ROTATION % len(dup_base):] + dup_base[:ROTATION % len(dup_base)]

# Search targets: some present, some absent
present_t = random.sample(ROTATED, min(5, N_ARR))
absent_t  = []
for _ in range(3):
    v = random.randint(1, VAL_RANGE)
    while v in set(ROTATED):
        v = random.randint(1, VAL_RANGE)
    absent_t.append(v)
TARGETS = present_t + absent_t

print(f"Rotated array parameters:")
print(f"  Array size      : {N_ARR}")
print(f"  Rotation offset : {ROTATION}")
print(f"  Value range     : 1 – {VAL_RANGE}")
print()
print(f"  Original sorted : {base_sorted}")
print(f"  After rotation  : {ROTATED}")
print(f"  True min index  : {ROTATED.index(min(ROTATED))}  (value {min(ROTATED)})")
print()
print(f"  Duplicates variant: {ROT_DUP}")
print(f"  Search targets    : {TARGETS}")


## Step 2 — Find minimum (rotation point) in rotated sorted array

In [ ]:
def find_min_rotated(arr):
    """
    Find the minimum element in a rotated sorted array (no duplicates).

    Key insight: in a rotated array one half is always properly sorted.
    Compare arr[mid] with arr[hi]:
      - If arr[mid] > arr[hi]: the minimum is in the RIGHT half (lo = mid + 1)
        because arr[mid] is in the larger-values portion that wraps around.
      - Else: minimum is in the LEFT half including mid (hi = mid)
        because arr[mid] <= arr[hi] means mid is in the ascending portion.
    When lo == hi we've found the minimum.
    O(log n).
    """
    lo, hi = 0, len(arr) - 1
    steps = 0
    trace = []
    while lo < hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        trace.append(f"  step {steps}: lo={lo} hi={hi} mid={mid} "
                      f"arr[mid]={arr[mid]} arr[hi]={arr[hi]} "
                      f"→ {'right' if arr[mid] > arr[hi] else 'left'}")
        if arr[mid] > arr[hi]:
            lo = mid + 1
        else:
            hi = mid
    return lo, steps, trace

def find_min_rotated_dup(arr):
    """
    Find minimum in rotated array WITH duplicates.
    When arr[mid] == arr[hi] we cannot determine which half has the minimum,
    so we safely shrink by one: hi -= 1.
    Worst case O(n) (all duplicates), average O(log n).
    """
    lo, hi = 0, len(arr) - 1
    steps = 0
    while lo < hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        if arr[mid] > arr[hi]:
            lo = mid + 1
        elif arr[mid] < arr[hi]:
            hi = mid
        else:
            hi -= 1    # ← can't tell which half — shrink by one
    return lo, steps

min_idx, min_steps, min_trace = find_min_rotated(ROTATED)
min_idx_dup, min_steps_dup    = find_min_rotated_dup(ROT_DUP)

print(f"Find minimum in rotated array (n={N_ARR}):")
print(f"  Array    : {ROTATED}")
print()
print(f"  Binary search trace:")
for t in min_trace:
    print(t)
print()
print(f"  Result   : index {min_idx},  value {ROTATED[min_idx]}")
print(f"  Correct  : {'✅ YES' if ROTATED[min_idx] == min(ROTATED) else '❌ NO'}")
print(f"  Steps    : {min_steps}  (log₂{N_ARR} ≈ {N_ARR.bit_length()})")
print()
print(f"  Duplicates variant: {ROT_DUP}")
print(f"  Min found : index {min_idx_dup},  value {ROT_DUP[min_idx_dup]}")
print(f"  Correct   : {'✅ YES' if ROT_DUP[min_idx_dup] == min(ROT_DUP) else '❌ NO'}")
print(f"  Steps     : {min_steps_dup}")


## Step 3 — Search target in rotated array

In [ ]:
def search_rotated(arr, target):
    """
    Search for target in rotated sorted array (no duplicates).

    At every step, ONE of the two halves is always properly sorted.
    Determine which half is sorted by comparing arr[lo] and arr[mid]:
      - If arr[lo] <= arr[mid]: LEFT half [lo..mid] is sorted.
        Check if target ∈ [arr[lo], arr[mid]]; if yes, search left, else right.
      - Else: RIGHT half [mid..hi] is sorted.
        Check if target ∈ [arr[mid], arr[hi]]; if yes, search right, else left.
    O(log n).
    """
    lo, hi = 0, len(arr) - 1
    steps = 0
    while lo <= hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        if arr[mid] == target:
            return mid, steps
        # Left half sorted?
        if arr[lo] <= arr[mid]:
            if arr[lo] <= target < arr[mid]:
                hi = mid - 1          # target in sorted left half
            else:
                lo = mid + 1          # target in right half
        else:
            # Right half sorted
            if arr[mid] < target <= arr[hi]:
                lo = mid + 1          # target in sorted right half
            else:
                hi = mid - 1          # target in left half
    return -1, steps

print(f"Search in rotated array (n={N_ARR}):")
print(f"  Array   : {ROTATED}")
print()
print(f"  {'Target':>8}  {'Index':>7}  {'Value':>7}  {'Steps':>7}  {'Correct':>8}")
print(f"  {'─'*8}  {'─'*7}  {'─'*7}  {'─'*7}  {'─'*8}")
all_correct = True
for t in TARGETS:
    idx, steps = search_rotated(ROTATED, t)
    expected   = ROTATED.index(t) if t in ROTATED else -1
    ok         = idx == expected
    if not ok: all_correct = False
    val_str    = str(ROTATED[idx]) if idx != -1 else "—"
    print(f"  {t:>8}  {idx:>7}  {val_str:>7}  {steps:>7}  {'✅' if ok else '❌':>8}")

print()
print(f"  All searches correct: {'✅ YES' if all_correct else '❌ NO'}")


## Step 4 — Exponential search (unbounded / stream scenario)

In [ ]:
def binary_search(arr, target):
    """Standard binary search — returns (index, steps)."""
    lo, hi = 0, len(arr) - 1
    steps = 0
    while lo <= hi:
        steps += 1
        mid = lo + (hi - lo) // 2
        if arr[mid] == target: return mid, steps
        elif arr[mid] < target: lo = mid + 1
        else: hi = mid - 1
    return -1, steps

def exponential_search(arr, target):
    """
    Exponential search: useful when array size is unknown or very large.

    1. Start with bound = 1.
    2. Double bound while arr[bound] < target (and bound < n).
    3. Binary search in [bound//2, min(bound, n-1)].

    Adapts to the scale of the answer — if target is near the start,
    bound stops early and binary search range is tiny.
    O(log i) where i is the target's index — faster than O(log n) for
    targets near the start of a large array.
    """
    n = len(arr)
    if arr[0] == target:
        return 0, 1
    bound = 1
    exp_steps = 0
    while bound < n and arr[bound] < target:
        bound *= 2
        exp_steps += 1
    # Binary search in [bound//2, min(bound, n-1)]
    lo, hi = bound // 2, min(bound, n - 1)
    bin_steps = 0
    while lo <= hi:
        bin_steps += 1
        mid = lo + (hi - lo) // 2
        if arr[mid] == target:
            return mid, exp_steps + bin_steps
        elif arr[mid] < target:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1, exp_steps + bin_steps

# Use fully sorted array for exponential search
SORTED_ARR = sorted(ROTATED)
exp_targets = random.sample(SORTED_ARR, min(6, len(SORTED_ARR)))

print(f"Exponential search on sorted array (n={N_ARR}):")
print(f"  Array : {SORTED_ARR}")
print()
print(f"  {'Target':>8}  {'Index':>7}  {'Steps':>7}  {'Bin-only steps':>15}  Notes")
print(f"  {'─'*8}  {'─'*7}  {'─'*7}  {'─'*15}  {'─'*30}")
for t in exp_targets:
    idx, exp_s = exponential_search(SORTED_ARR, t)
    _, bin_s   = binary_search(SORTED_ARR, t)
    pos_note   = "near start" if idx < N_ARR // 4 else ("near end" if idx > 3*N_ARR//4 else "middle")
    print(f"  {t:>8}  {idx:>7}  {exp_s:>7}  {bin_s:>15}  {pos_note}")

print()
print(f"  Note: exponential search is fastest when target is near the beginning.")


## 📋 Final Report

In [ ]:
W = 58
print("╔" + "═"*W + "╗")
print("║" + " ROTATED ARRAY NAVIGATOR — REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<28}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<28}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  ROTATED ARRAY" + " "*(W-15) + "║")
print(f"║  {'Array size (n)':<28}: {N_ARR:<26}║")
print(f"║  {'Rotation offset':<28}: {ROTATION:<26}║")
print(f"║  {'True minimum value':<28}: {min(ROTATED):<26}║")
print(f"║  {'Min found (no dups)':<28}: {ROTATED[min_idx]} at index {min_idx}{'':<15}║")
print(f"║  {'Steps (no dups)':<28}: {min_steps:<26}║")
print(f"║  {'Steps (with dups)':<28}: {min_steps_dup:<26}║")
print("╠" + "═"*W + "╣")
print("║  SEARCH IN ROTATED ARRAY" + " "*(W-25) + "║")
print(f"║  {'Targets searched':<28}: {len(TARGETS):<26}║")
print(f"║  {'All correct':<28}: {'YES ✅' if all_correct else 'NO ❌':<26}║")
print("╠" + "═"*W + "╣")
print("║  EXPONENTIAL SEARCH" + " "*(W-20) + "║")
print(f"║  {'Array size':<28}: {N_ARR:<26}║")
print(f"║  {'Targets searched':<28}: {len(exp_targets):<26}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about searching in this context?

*Your answer here:*

---

**Q2 — Why is binary search O(log n) and not O(n)?**
Explain in your own words why halving the search space each step leads to logarithmic complexity,
and give one example from your results that illustrates this.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
